## 1. Install Required Packages

Run this cell first to install all dependencies.

In [ ]:
# Install required packages
!pip install -q diffusers transformers accelerate torch torchvision pillow huggingface_hub

## 2. Hugging Face Authentication

Login to Hugging Face to access Stable Diffusion models.

**Get your token:** https://huggingface.co/settings/tokens

In [ ]:
from huggingface_hub import login

# Replace with your Hugging Face token
HF_TOKEN = "hf_YOUR_TOKEN_HERE"  # Get from https://huggingface.co/settings/tokens

# Login to Hugging Face
login(token=HF_TOKEN)
print("✅ Successfully logged in to Hugging Face!")

## 3. Import Libraries and Setup

In [ ]:
import torch
from diffusers import DiffusionPipeline, DPMSolverMultistepScheduler
from PIL import Image
import os
from datetime import datetime

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"CUDA memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

## 4. Load Stable Diffusion Model

This will download the model (~7GB) on first run. Subsequent runs will use cached version.

In [ ]:
# Determine device (GPU or CPU)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🔧 Using device: {device}")

# Load Stable Diffusion XL model
print("📦 Loading Stable Diffusion XL model...")
print("   (This may take a few minutes on first run)")

pipe = DiffusionPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    use_safetensors=True,
    variant="fp16" if device == "cuda" else None
)

# Optimize for speed
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)

# Move to device
pipe = pipe.to(device)

# Enable memory optimizations if on GPU
if device == "cuda":
    pipe.enable_attention_slicing()
    pipe.enable_vae_slicing()

print("✅ Model loaded successfully!")

## 5. Create Output Directory

In [ ]:
# Create directory for generated images
output_dir = "generated_images"
os.makedirs(output_dir, exist_ok=True)
print(f"📁 Output directory: {os.path.abspath(output_dir)}")

## 6. Generate Your First Image! 🎨

Customize the prompt below to generate images for your marketing campaigns.

In [ ]:
# Your creative prompt
prompt = "A modern restaurant interior with warm lighting, cozy atmosphere, elegant table settings, professional photography, 4k quality"

# Optional: Negative prompt (what to avoid)
negative_prompt = "blurry, low quality, distorted, ugly, bad anatomy"

# Generation parameters
num_inference_steps = 30  # Higher = better quality but slower (20-50 recommended)
guidance_scale = 7.5      # How closely to follow prompt (7-10 recommended)
seed = 42                 # Set for reproducibility, or None for random

print(f"🎨 Generating image...")
print(f"   Prompt: {prompt}")
print(f"   Steps: {num_inference_steps}, Guidance: {guidance_scale}")

# Generate image
generator = torch.Generator(device=device).manual_seed(seed) if seed else None

image = pipe(
    prompt=prompt,
    negative_prompt=negative_prompt,
    num_inference_steps=num_inference_steps,
    guidance_scale=guidance_scale,
    generator=generator
).images[0]

# Display the image
display(image)

# Save with timestamp
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
filename = f"{output_dir}/generated_{timestamp}.png"
image.save(filename)

print(f"✅ Image generated and saved: {filename}")

## 7. Generate Multiple Variations

Create multiple versions with different seeds for A/B testing.

In [ ]:
# Generate 3 variations of the same prompt
prompt = "Social media post for a restaurant special offer, vibrant colors, mouth-watering food, professional food photography"
num_variations = 3

print(f"🎨 Generating {num_variations} variations...\n")

images = []
for i in range(num_variations):
    print(f"Generating variation {i+1}/{num_variations}...")
    
    # Different seed for each variation
    generator = torch.Generator(device=device).manual_seed(42 + i)
    
    image = pipe(
        prompt=prompt,
        negative_prompt="blurry, low quality, distorted",
        num_inference_steps=25,
        guidance_scale=7.5,
        generator=generator
    ).images[0]
    
    images.append(image)
    
    # Save each variation
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"{output_dir}/variation_{i+1}_{timestamp}.png"
    image.save(filename)
    print(f"   Saved: {filename}")

print("\n✅ All variations generated!\n")

# Display all variations
for i, img in enumerate(images, 1):
    print(f"Variation {i}:")
    display(img)

## 8. RAAMP Marketing Content Generator

Pre-configured prompts for common RAAMP marketing needs.

In [ ]:
# RAAMP Marketing Prompt Templates
marketing_prompts = {
    "restaurant_interior": "Modern restaurant interior, warm ambient lighting, elegant table settings, cozy atmosphere, professional interior photography, high quality, 4k",
    "food_hero": "Delicious gourmet dish, professional food photography, appetizing presentation, vibrant colors, garnished beautifully, restaurant quality, commercial photography",
    "social_post": "Eye-catching social media post design, vibrant colors, modern minimalist style, marketing campaign, professional graphic design, clean layout",
    "promotional_banner": "Restaurant promotional banner, special offer design, attention-grabbing, bold typography, appetizing food imagery, professional marketing material",
    "brand_lifestyle": "Lifestyle shot of people enjoying restaurant dining, happy customers, social gathering, warm atmosphere, candid photography, authentic moment",
    "menu_item": "Restaurant menu item showcase, close-up food photography, detailed textures, professional lighting, appetizing presentation, commercial quality"
}

# Select a template
template_name = "food_hero"  # Change this to generate different types
selected_prompt = marketing_prompts[template_name]

# Optional: Customize the prompt
custom_addition = "pizza"  # Add specific details
final_prompt = f"{selected_prompt}, featuring {custom_addition}"

print(f"🎯 Generating: {template_name}")
print(f"📝 Prompt: {final_prompt}\n")

# Generate
image = pipe(
    prompt=final_prompt,
    negative_prompt="blurry, low quality, distorted, ugly, bad composition",
    num_inference_steps=30,
    guidance_scale=7.5,
    generator=torch.Generator(device=device).manual_seed(42)
).images[0]

display(image)

# Save
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
filename = f"{output_dir}/{template_name}_{timestamp}.png"
image.save(filename)
print(f"\n✅ Saved: {filename}")

## 9. Batch Generation for Campaign

Generate multiple images for a complete marketing campaign.

In [ ]:
# Define your campaign prompts
campaign_prompts = [
    "Restaurant special discount banner, 50% off text, vibrant red and yellow colors, appetizing food in background",
    "Happy hour promotion poster, cocktails and drinks, evening atmosphere, professional bar photography",
    "Weekend brunch special, fresh breakfast items, morning light, cozy cafe setting, inviting atmosphere"
]

print(f"🚀 Generating {len(campaign_prompts)} campaign images...\n")

campaign_images = []
for idx, prompt in enumerate(campaign_prompts, 1):
    print(f"[{idx}/{len(campaign_prompts)}] Generating: {prompt[:60]}...")
    
    image = pipe(
        prompt=prompt,
        negative_prompt="blurry, low quality, distorted",
        num_inference_steps=25,
        guidance_scale=7.5,
        generator=torch.Generator(device=device).manual_seed(100 + idx)
    ).images[0]
    
    campaign_images.append(image)
    
    # Save
    filename = f"{output_dir}/campaign_{idx}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.png"
    image.save(filename)
    print(f"   ✅ Saved: {filename}\n")

print("🎉 Campaign generation complete!\n")

# Display all campaign images
for idx, img in enumerate(campaign_images, 1):
    print(f"Campaign Image {idx}:")
    display(img)

## 10. Advanced: Custom Image Size

Generate images in specific dimensions for different platforms.

In [ ]:
# Platform-specific dimensions
dimensions = {
    "instagram_square": (1024, 1024),    # 1:1
    "instagram_portrait": (1024, 1280),  # 4:5
    "facebook_cover": (1200, 630),       # ~2:1
    "story": (1080, 1920)                # 9:16
}

# Select format
format_name = "instagram_square"
width, height = dimensions[format_name]

prompt = "Modern restaurant ambiance, professional photography, high quality"

print(f"📐 Generating {format_name} ({width}x{height})...\n")

image = pipe(
    prompt=prompt,
    negative_prompt="blurry, low quality",
    num_inference_steps=30,
    guidance_scale=7.5,
    width=width,
    height=height,
    generator=torch.Generator(device=device).manual_seed(42)
).images[0]

display(image)

filename = f"{output_dir}/{format_name}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.png"
image.save(filename)
print(f"\n✅ Saved: {filename}")

## 11. Quick Generation Function

Simple function for easy image generation.

In [ ]:
def generate_marketing_image(
    prompt: str,
    save_name: str = None,
    steps: int = 25,
    guidance: float = 7.5,
    seed: int = None
):
    """
    Quick image generation function.
    
    Args:
        prompt: Your creative prompt
        save_name: Custom filename (optional)
        steps: Number of inference steps (20-50)
        guidance: Guidance scale (7-10)
        seed: Random seed for reproducibility
    
    Returns:
        PIL Image object
    """
    print(f"🎨 Generating: {prompt[:60]}...")
    
    generator = torch.Generator(device=device).manual_seed(seed) if seed else None
    
    image = pipe(
        prompt=prompt,
        negative_prompt="blurry, low quality, distorted",
        num_inference_steps=steps,
        guidance_scale=guidance,
        generator=generator
    ).images[0]
    
    # Save
    if save_name:
        filename = f"{output_dir}/{save_name}.png"
    else:
        filename = f"{output_dir}/generated_{datetime.now().strftime('%Y%m%d_%H%M%S')}.png"
    
    image.save(filename)
    print(f"✅ Saved: {filename}\n")
    
    return image

# Example usage
img = generate_marketing_image(
    prompt="Delicious pizza with melted cheese, professional food photography",
    save_name="pizza_hero",
    seed=42
)

display(img)